# Dataset Expansion Notebook

This notebook expands a wide dataset from 680 to 10,000 columns using various transformations:
- Positive and negative correlations
- Column combinations
- Noise injection
- Scaling transformations

All generated columns maintain realistic stock-like values.

In [1]:
import pandas as pd
import numpy as np
import random
import string
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [2]:
def generate_random_column_name(length=6):
    """Generate random stock-like column names"""
    prefixes = ['STOCK', 'ASSET', 'FUND', 'BOND', 'REIT', 'ETF', 'INDEX', 'COMP']
    suffixes = ['_RET', '_VOL', '_BETA', '_PRICE', '_YIELD', '_CAP', '_RATIO', '_SCORE']
    
    if random.random() < 0.6:
        # Stock ticker style
        ticker = ''.join(random.choices(string.ascii_uppercase, k=random.randint(3, 5)))
        if random.random() < 0.3:
            ticker += random.choice(suffixes)
        return ticker
    else:
        # Descriptive name style
        prefix = random.choice(prefixes)
        number = random.randint(1, 999)
        suffix = random.choice(suffixes) if random.random() < 0.5 else ''
        return f"{prefix}_{number}{suffix}"

In [3]:
def normalize_to_stock_range(data, min_val=1.0, max_val=1000.0):
    """Normalize data to typical stock price range preserving temporal structure"""
    # Handle edge case where all values are the same
    if data.max() == data.min():
        return np.full_like(data, (min_val + max_val) / 2)

    # Use percentile-based normalization to reduce impact of outliers
    p5, p95 = np.percentile(data, [5, 95])
    data_clipped = np.clip(data, p5, p95)

    data_norm = (data_clipped - data_clipped.min()) / (data_clipped.max() - data_clipped.min())
    return data_norm * (max_val - min_val) + min_val

In [4]:
def create_correlated_features(base_data, n_features, correlation_strength=0.7, used_names=None, get_unique_name=None):
    """Create features with specified correlation to base data"""
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Choose random base column
        base_col = np.random.choice(base_data.columns)
        base_values = base_data[base_col].values
        
        # Generate noise
        noise = np.random.normal(0, 1, len(base_values))
        
        # Create correlated feature
        if correlation_strength > 0:
            # Positive correlation
            new_values = correlation_strength * base_values + np.sqrt(1 - correlation_strength**2) * noise
        else:
            # Negative correlation
            new_values = abs(correlation_strength) * (-base_values) + np.sqrt(1 - correlation_strength**2) * noise
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = get_unique_name() if get_unique_name else generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [5]:
def create_combined_features(base_data, n_features, used_names=None, get_unique_name=None):
    """Create features by combining existing columns"""
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Choose 2-4 random columns to combine
        n_cols = np.random.randint(2, 5)
        selected_cols = np.random.choice(base_data.columns, n_cols, replace=False)
        
        # Choose combination method
        method = np.random.choice(['weighted_sum', 'product', 'ratio', 'difference'])
        
        if method == 'weighted_sum':
            weights = np.random.uniform(-1, 1, n_cols)
            new_values = np.sum([w * base_data[col].values for w, col in zip(weights, selected_cols)], axis=0)
        
        elif method == 'product':
            new_values = np.prod([base_data[col].values for col in selected_cols], axis=0)
            new_values = np.sign(new_values) * np.log1p(np.abs(new_values))
        
        elif method == 'ratio':
            if n_cols >= 2:
                num = base_data[selected_cols[0]].values
                denom = base_data[selected_cols[1]].values + 0.001  # Avoid division by zero
                new_values = num / denom
            else:
                new_values = base_data[selected_cols[0]].values
        
        elif method == 'difference':
            if n_cols >= 2:
                new_values = base_data[selected_cols[0]].values - base_data[selected_cols[1]].values
            else:
                new_values = base_data[selected_cols[0]].values
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = get_unique_name() if get_unique_name else generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [6]:
def create_pca_features(base_data, n_components, n_features, used_names=None, get_unique_name=None):
    """Create features using PCA components"""
    pca = PCA(n_components=n_components)
    pca_data = pca.fit_transform(base_data)
    
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Select random PCA components and weights
        n_comp_selected = np.random.randint(1, min(n_components, 5) + 1)
        selected_components = np.random.choice(n_components, n_comp_selected, replace=False)
        weights = np.random.uniform(-1, 1, n_comp_selected)
        
        # Create new feature as weighted sum of PCA components
        new_values = np.sum([w * pca_data[:, comp] for w, comp in zip(weights, selected_components)], axis=0)
        
        # Add some noise
        noise = np.random.normal(0, 0.1, len(new_values))
        new_values = new_values + noise
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = get_unique_name() if get_unique_name else generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [7]:
def create_noise_features(base_data, n_features, used_names=None, get_unique_name=None):
    """Create features with controlled noise patterns that mimic stock-like temporal behavior"""
    new_features = pd.DataFrame()

    for i in range(n_features):
        n_samples = len(base_data)

        # Start with a base reference stock to get realistic temporal patterns
        base_col = np.random.choice(base_data.columns)
        base_values = base_data[base_col].values

        # Calculate returns from base stock
        base_returns = np.diff(np.log(base_values + 1e-10))
        base_returns = np.concatenate([[0], base_returns])

        # Generate noise with similar volatility structure
        noise_scale = np.random.uniform(0.3, 1.5)  # Random volatility scaling
        noise = np.random.normal(0, np.std(base_returns) * noise_scale, n_samples)

        # Smooth the noise to reduce high-frequency oscillations
        window_size = np.random.randint(3, 8)
        noise_smooth = np.convolve(noise, np.ones(window_size)/window_size, mode='same')

        # Create synthetic returns by blending base returns with smooth noise
        blend_factor = np.random.uniform(0.1, 0.3)  # Low blend keeps temporal structure
        synthetic_returns = blend_factor * base_returns + (1 - blend_factor) * noise_smooth

        # Convert returns to price series
        initial_price = np.random.uniform(5, 200)  # Random starting price
        log_prices = np.cumsum(synthetic_returns) + np.log(initial_price)
        new_values = np.exp(log_prices)

        # Ensure reasonable stock price range
        new_values = np.clip(new_values, 0.5, 2000)

        col_name = get_unique_name() if get_unique_name else generate_random_column_name()
        new_features[col_name] = new_values

    return new_features

In [8]:
def expand_dataset(data, target_columns=10000):
    """Main function to expand dataset from 680 to target number of columns"""
    print(f"Starting with {data.shape[1]} columns")
    print(f"Target: {target_columns} columns")
    print(f"Preserving date index: {data.index.name}")
    
    # Keep the original date index
    expanded_data = data.copy()
    columns_to_add = target_columns - data.shape[1]
    
    # Track used column names to prevent duplicates
    used_names = set(data.columns)
    
    def get_unique_column_name():
        """Generate unique column name not in used_names"""
        max_attempts = 1000
        for _ in range(max_attempts):
            name = generate_random_column_name()
            if name not in used_names:
                used_names.add(name)
                return name
        # If still can't find unique name after max_attempts, use numbered fallback
        i = 0
        while True:
            name = f"SYNTH_{i}"
            if name not in used_names:
                used_names.add(name)
                return name
            i += 1
    
    # Distribution of new features
    n_positive_corr = int(columns_to_add * 0.25)  # 25% positive correlations
    n_negative_corr = int(columns_to_add * 0.25)  # 25% negative correlations
    n_combined = int(columns_to_add * 0.30)       # 30% combined features
    n_pca = int(columns_to_add * 0.10)            # 10% PCA features
    n_noise = columns_to_add - (n_positive_corr + n_negative_corr + n_combined + n_pca)  # Remaining as noise
    
    print(f"Creating {n_positive_corr} positive correlation features...")
    pos_corr_features = create_correlated_features(data, n_positive_corr, 0.7, used_names, get_unique_column_name)
    # Preserve the date index
    pos_corr_features.index = data.index
    expanded_data = pd.concat([expanded_data, pos_corr_features], axis=1)
    
    print(f"Creating {n_negative_corr} negative correlation features...")
    neg_corr_features = create_correlated_features(data, n_negative_corr, -0.6, used_names, get_unique_column_name)
    neg_corr_features.index = data.index
    expanded_data = pd.concat([expanded_data, neg_corr_features], axis=1)
    
    print(f"Creating {n_combined} combined features...")
    combined_features = create_combined_features(data, n_combined, used_names, get_unique_column_name)
    combined_features.index = data.index
    expanded_data = pd.concat([expanded_data, combined_features], axis=1)
    
    print(f"Creating {n_pca} PCA-based features...")
    pca_features = create_pca_features(data, min(50, data.shape[1]//2), n_pca, used_names, get_unique_column_name)
    pca_features.index = data.index
    expanded_data = pd.concat([expanded_data, pca_features], axis=1)
    
    print(f"Creating {n_noise} noise features...")
    noise_features = create_noise_features(data, n_noise, used_names, get_unique_column_name)
    noise_features.index = data.index
    expanded_data = pd.concat([expanded_data, noise_features], axis=1)
    
    print(f"Final dataset shape: {expanded_data.shape}")
    print(f"Unique columns: {len(expanded_data.columns.unique())} (should match {expanded_data.shape[1]})")
    print(f"Date index preserved: {expanded_data.index.name}")
    print(f"Date range: {expanded_data.index[0]} to {expanded_data.index[-1]}")
    return expanded_data

## Example Usage

Load your wide dataset and expand it:

In [9]:
# Load the original wide dataset to get the timestamps
original_data_path = '../../data/etfs_close.pkl'  # or your actual wide dataset
original_stocks_path = '../../data/stocks_adjclose.pkl'

# Load original data to get the proper date index
stocks_data = pd.read_pickle(original_stocks_path)
etfs_data = pd.read_pickle(original_data_path)

# Create wide dataset like in the portfolio optimization notebook
wide_data = pd.merge(stocks_data, etfs_data, on='ds', how='left').set_index('ds').dropna()

print(f"Original wide dataset shape: {wide_data.shape}")
print(f"Date range: {wide_data.index[0]} to {wide_data.index[-1]}")
print(f"Sample data range: {wide_data.min().min():.4f} to {wide_data.max().max():.4f}")

# Use the wide dataset as the base for expansion
sample_data = wide_data.copy()

print(f"Base data shape: {sample_data.shape}")
print(f"Index type: {type(sample_data.index)}")
print(f"First few dates: {sample_data.index[:5].tolist()}")

Original wide dataset shape: (3521, 680)
Date range: 2011-01-03 00:00:00 to 2024-12-30 00:00:00
Sample data range: 0.2609 to 44802.2070
Base data shape: (3521, 680)
Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
First few dates: [Timestamp('2011-01-03 00:00:00'), Timestamp('2011-01-04 00:00:00'), Timestamp('2011-01-05 00:00:00'), Timestamp('2011-01-06 00:00:00'), Timestamp('2011-01-07 00:00:00')]


In [10]:
print("Regenerating dataset with proper price scaling...")

# Expand the dataset with proper stock price scaling
expanded_dataset = expand_dataset(sample_data, target_columns=1200)

print("\nDataset expansion completed!")
print(f"Original shape: {sample_data.shape}")
print(f"Expanded shape: {expanded_dataset.shape}")
print(f"Value range: {expanded_dataset.min().min():.4f} to {expanded_dataset.max().max():.4f}")
print(f"Original data range: {sample_data.min().min():.4f} to {sample_data.max().max():.4f}")

# Check synthetic columns specifically (excluding original data)
synthetic_cols = expanded_dataset.columns[len(sample_data.columns):]
if len(synthetic_cols) > 0:
    synthetic_data = expanded_dataset[synthetic_cols]
    print(f"Synthetic data range: {synthetic_data.min().min():.4f} to {synthetic_data.max().max():.4f}")
    print(f"Synthetic data mean: {synthetic_data.mean().mean():.4f}")
    print(f"Synthetic data std: {synthetic_data.std().mean():.4f}")

Regenerating dataset with proper price scaling...
Starting with 680 columns
Target: 1200 columns
Preserving date index: ds
Creating 130 positive correlation features...
Creating 130 negative correlation features...
Creating 156 combined features...
Creating 52 PCA-based features...
Creating 52 noise features...
Final dataset shape: (3521, 1200)
Unique columns: 1200 (should match 1200)
Date index preserved: ds
Date range: 2011-01-03 00:00:00 to 2024-12-30 00:00:00

Dataset expansion completed!
Original shape: (3521, 680)
Expanded shape: (3521, 1200)
Value range: 0.2609 to 44802.2070
Original data range: 0.2609 to 44802.2070
Synthetic data range: 1.0000 to 1156.5410
Synthetic data mean: 463.6956
Synthetic data std: 273.0002


In [11]:
# Display some statistics
print("Sample column names from expanded dataset:")
print(expanded_dataset.columns[-20:].tolist())  # Last 20 column names

print("\nBasic statistics:")
print(expanded_dataset.describe().iloc[:, -5:])  # Stats for last 5 columns

Sample column names from expanded dataset:
['FDSXV', 'SDCNJ_RATIO', 'HDEY', 'ETF_179', 'UFPEH', 'ASOWU', 'XZFO', 'CJIQZ', 'ETF_458_PRICE', 'JUS_SCORE', 'FUND_906_RATIO', 'CMWQ', 'UAA_VOL', 'XEB', 'ETF_609', 'STOCK_442', 'ECFE', 'REIT_677_PRICE', 'OVDJ', 'QMCFQ']

Basic statistics:


         STOCK_442         ECFE  REIT_677_PRICE         OVDJ        QMCFQ
count  3521.000000  3521.000000     3521.000000  3521.000000  3521.000000
mean    124.197487   181.289716       20.836405   112.447049   273.696976
std      15.057114    28.050934        7.718196    54.888291    51.926295
min      90.935684   125.538872        8.085893    41.115251   170.400035
25%     111.813237   155.778320       14.354761    73.638953   228.028169
50%     125.852546   178.270114       20.000462    85.795327   273.457023
75%     136.964259   206.130404       27.035927   142.925829   318.568255
max     154.102162   237.606816       36.184017   271.691095   384.743444


In [12]:
# Save the updated expanded dataset to data folder as synthetic_close.pkl
import os
os.makedirs('../../data', exist_ok=True)

# Ensure the dataset has the proper date index named 'ds'
expanded_dataset.index.name = 'ds'

# Reset index to make 'ds' a column, then set it back as index to ensure proper format
expanded_dataset_with_ds = expanded_dataset.reset_index()
print(f"Dataset with ds column shape: {expanded_dataset_with_ds.shape}")
print(f"Columns include 'ds': {'ds' in expanded_dataset_with_ds.columns}")
print(f"Date range: {expanded_dataset_with_ds['ds'].min()} to {expanded_dataset_with_ds['ds'].max()}")

# Check value ranges after update
print(f"Updated value range: {expanded_dataset_with_ds.select_dtypes(include=[np.number]).min().min():.4f} to {expanded_dataset_with_ds.select_dtypes(include=[np.number]).max().max():.4f}")

# Save with 'ds' as a column (like the original datasets)
expanded_dataset_with_ds.to_pickle('../../data/synthetic_close.pkl')
print("Updated expanded dataset saved as 'quantum_portfolio/data/synthetic_close.pkl'")

# Verify what we saved
test_load = pd.read_pickle('../../data/synthetic_close.pkl')
print(f"Verification - loaded dataset shape: {test_load.shape}")
print(f"Verification - has 'ds' column: {'ds' in test_load.columns}")
if 'ds' in test_load.columns:
    print(f"Verification - ds column type: {test_load['ds'].dtype}")
    print(f"Verification - first few ds values: {test_load['ds'].head().tolist()}")

# Check final value range
numeric_cols = test_load.select_dtypes(include=[np.number]).columns
print(f"Final verification - value range: {test_load[numeric_cols].min().min():.4f} to {test_load[numeric_cols].max().max():.4f}")

print("Dataset regeneration completed successfully!")
print(f"Successfully updated synthetic dataset with proper price scaling")

Dataset with ds column shape: (3521, 1201)
Columns include 'ds': True
Date range: 2011-01-03 00:00:00 to 2024-12-30 00:00:00
Updated value range: 0.2609 to 44802.2070
Updated expanded dataset saved as 'quantum_portfolio/data/synthetic_close.pkl'
Verification - loaded dataset shape: (3521, 1201)
Verification - has 'ds' column: True
Verification - ds column type: datetime64[ns]
Verification - first few ds values: [Timestamp('2011-01-03 00:00:00'), Timestamp('2011-01-04 00:00:00'), Timestamp('2011-01-05 00:00:00'), Timestamp('2011-01-06 00:00:00'), Timestamp('2011-01-07 00:00:00')]
Final verification - value range: 0.2609 to 44802.2070
Dataset regeneration completed successfully!
Successfully updated synthetic dataset with proper price scaling


In [13]:
expanded_dataset_with_ds.head()

,ds,A,AAPL,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,FUND_906_RATIO,CMWQ,UAA_VOL,XEB,ETF_609,STOCK_442,ECFE,REIT_677_PRICE,OVDJ,QMCFQ
0,2011-01-03,26.781836,9.917951,16.942663,9.349445,37.585785,31.290001,27.436106,20.712511,29.848969,...,18.107629,75.492461,129.462925,125.937036,87.798646,98.343111,125.783745,17.166407,145.878215,190.952276
1,2011-01-04,26.532440,9.969709,17.102097,9.291334,37.338249,31.510000,27.125227,20.698887,29.741137,...,18.030430,75.850052,128.858985,124.957257,87.413089,98.193005,125.538872,17.047364,145.654602,191.300773
2,2011-01-05,26.474880,10.051261,17.102097,9.305071,37.346004,32.220001,27.183071,20.794275,30.216925,...,17.971160,76.258827,128.802213,124.441944,87.940231,97.989877,125.854862,16.999914,146.992259,191.298014
3,2011-01-06,26.526041,10.043136,17.066668,9.179339,37.485237,32.270000,27.334898,21.591436,30.451658,...,17.821023,76.472465,129.117381,124.334951,87.966434,97.917770,125.800167,17.030518,148.804873,190.944464
4,2011-01-07,26.615564,10.115060,17.137522,9.109608,37.547104,32.040001,27.175838,21.768579,30.521458,...,17.680604,76.430398,129.274692,124.344211,88.868399,97.893132,126.111993,17.036943,151.627367,189.739391


## Notes

- All generated features maintain stock-like value ranges (typically -15% to +15%)
- Column names are randomly generated in stock ticker style
- Different correlation levels and combination methods ensure diversity
- The expansion maintains statistical properties suitable for financial analysis
- Random seeds are set for reproducibility